# Modul 2: Backpropagation dan Automatic Differentiation

**Nama:** Ahmad Rizqi  
**NIM:**  122450138
**Kelas:** RA  
**Tanggal:** 2026-09-27  

Simpan berkas ini sebagai `M02_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Nilai Kasus 1 tidak boleh diubah; seluruh angka pada modul mengacu padanya.
3. Gunakan `float64` untuk semua perhitungan gradien.
4. Tuliskan turunan manual pada sel markdown, bukan hanya di kertas.
5. Notebook harus lolos *Restart Kernel and Run All* sebelum dikumpulkan.
6. Luaran: `M02_NIM.ipynb`, `M02_NIM.pdf`, dan `M02_NIM_metrics.csv`.

In [4]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch import nn

NIM = '122450138' 
SEED = int(str(NIM)[-9:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)
pd.set_option('display.precision', 8)
environment = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'device': str(DEVICE),
    'seed': SEED,
    'default_dtype': str(torch.get_default_dtype()),
}
environment

{'python': '3.11.4',
 'numpy': '2.4.6',
 'pandas': '3.0.6',
 'torch': '2.14.0+cpu',
 'device': 'cpu',
 'seed': 122450138,
 'default_dtype': 'torch.float64'}

## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

1. **Gradien lokal vs gradien total pada satu simpul:** TODO
2. **Mengapa `backward()` hanya dapat dipanggil pada tensor skalar:** TODO
3. **Isi `.grad` bila `backward()` dipanggil dua kali tanpa `zero_grad()`:** TODO
4. **Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$, bukan $-y/p$:** TODO

**Graf komputasi Kasus 1.** Tuliskan urutan simpul dari $\mathbf{x}$ sampai $\mathcal{L}$, lalu tandai gradien lokal di setiap simpul:

TODO

## B. Turunan manual - 20 poin

Kasus 1 memakai nilai tetap berikut:

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1$$

Tuliskan penurunan Anda **berurutan** di sini, satu baris satu langkah:

1. $\partial\mathcal{L}/\partial z^{(2)} =$ TODO
2. $\partial\mathcal{L}/\partial \mathbf{W}^{(2)} =$ TODO
3. $\partial\mathcal{L}/\partial b^{(2)} =$ TODO
4. $\partial\mathcal{L}/\partial \mathbf{h} =$ TODO
5. $\partial\mathcal{L}/\partial \mathbf{z}^{(1)} =$ TODO
6. $\partial\mathcal{L}/\partial \mathbf{W}^{(1)} =$ TODO
7. $\partial\mathcal{L}/\partial \mathbf{b}^{(1)} =$ TODO

Cantumkan pula shape setiap gradien: TODO

In [5]:
x = np.array([2.0, -1.0], dtype=np.float64)
W1 = np.array([[0.5, -0.5], [1.0, 1.0]], dtype=np.float64)
b1 = np.array([0.0, 0.0], dtype=np.float64)
W2 = np.array([2.0, -1.0], dtype=np.float64)
b2 = np.float64(0.5)
y = np.float64(1.0)

def sigmoid_stabil(z):
    z = np.asarray(z, dtype=np.float64)
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

def forward(x, W1, b1, W2, b2, y):
    z1 = W1 @ x + b1
    h = np.maximum(z1, 0.0)
    z2 = W2 @ h + b2
    p = sigmoid_stabil(z2).item()
    # Bentuk BCE-with-logits yang stabil: softplus(z2) - y*z2.
    loss = np.logaddexp(0.0, z2) - y * z2
    return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}

nilai = forward(x, W1, b1, W2, b2, y)
print({k: np.round(v, 6) for k, v in nilai.items()})

# Pemeriksaan wajib: tidak diubah.
assert np.allclose(nilai['z1'], [1.5, 1.0]), 'z1 belum benar'
assert np.isclose(nilai['z2'], 2.5), 'logit belum benar'
assert np.isclose(nilai['loss'], 0.0788897, atol=1e-6), 'loss belum benar'
print('forward pass sesuai Kasus 1')

{'z1': array([1.5, 1. ]), 'h': array([1.5, 1. ]), 'z2': np.float64(2.5), 'p': np.float64(0.924142), 'loss': np.float64(0.07889)}
forward pass sesuai Kasus 1


In [6]:
def backward(x, W1, W2, y, nilai):
    z1, h, p = nilai['z1'], nilai['h'], nilai['p']
    dz2 = p - y
    dW2 = dz2 * h
    db2 = dz2
    dh = dz2 * W2
    dz1 = dh * (z1 > 0)
    dW1 = np.outer(dz1, x)
    db1 = dz1
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

grad_manual = backward(x, W1, W2, y, nilai)
for nama, value in grad_manual.items():
    print(f'{nama:>3}: {np.round(value, 7)}')


assert np.allclose(grad_manual['W2'], [-0.1137873, -0.0758582], atol=1e-6)
assert grad_manual['W1'].shape == W1.shape, 'shape dW1 harus sama dengan W1'
print('gradien manual sesuai angka acuan')

 W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
 b1: [-0.1517164  0.0758582]
 W2: [-0.1137873 -0.0758582]
 b2: -0.0758582
gradien manual sesuai angka acuan


## C. Autograd - 20 poin

Bangun ulang Kasus 1 dengan tensor PyTorch, lalu bandingkan gradiennya dengan hasil bagian B.

In [7]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)
tx, ty = torch.tensor(x), torch.tensor(y)
kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
    tz1 = tW1 @ tx + tb1
    th = torch.relu(tz1)
    tz2 = tW2 @ th + tb2
    return kriteria(tz2, ty)

loss = forward_torch()
loss.backward()

for nama, tensor in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    selisih = np.max(np.abs(tensor.grad.detach().numpy() - grad_manual[nama]))
    print(f'{nama}: autograd = {np.round(tensor.grad.detach().numpy(), 7)}   selisih maks = {selisih:.2e}')
    assert selisih < 1e-10, f'gradien {nama} tidak sesuai dengan hasil manual'
print('autograd cocok dengan backward manual')

W1: autograd = [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]   selisih maks = 0.00e+00
b1: autograd = [-0.1517164  0.0758582]   selisih maks = 0.00e+00
W2: autograd = [-0.1137873 -0.0758582]   selisih maks = 0.00e+00
b2: autograd = -0.0758582   selisih maks = 0.00e+00
autograd cocok dengan backward manual


In [8]:
grad_W2_satu = tW2.grad.detach().clone()
forward_torch().backward()  
grad_W2_dua = tW2.grad.detach().clone()
print('satu backward        :', np.round(grad_W2_satu.numpy(), 7))
print('dua backward         :', np.round(grad_W2_dua.numpy(), 7))
assert torch.allclose(grad_W2_dua, 2 * grad_W2_satu)

for tensor in (tW1, tb1, tW2, tb2):
    tensor.grad = None
forward_torch().backward()
grad_W2_reset = tW2.grad.detach().clone()
print('setelah grad dihapus :', np.round(grad_W2_reset.numpy(), 7))
assert torch.allclose(grad_W2_reset, grad_W2_satu)

satu backward        : [-0.1137873 -0.0758582]
dua backward         : [-0.2275745 -0.1517164]
setelah grad dihapus : [-0.1137873 -0.0758582]


**Penjelasan akumulasi gradien:** Ketika `backward()` dijalankan untuk kedua kalinya tanpa mengosongkan nilai `.grad`, diperoleh `tW2.grad = [-0.2275745, -0.1517164]`. Nilai tersebut merupakan **dua kali lipat** dari gradien yang dihasilkan oleh satu kali proses `backward()`, yaitu `[-0.1137873, -0.0758582]`. Setelah atribut `.grad` diatur menjadi `None`, proses `backward()` berikutnya kembali menghasilkan nilai gradien seperti semula. Hal ini menunjukkan bahwa PyTorch secara default melakukan akumulasi gradien. Oleh karena itu, karena optimizer menggunakan nilai yang tersimpan pada `.grad` ketika menjalankan `step()`, proses training umumnya diawali dengan `optimizer.zero_grad()` sebelum melakukan perhitungan backward pada batch berikutnya. Dengan demikian, gradien dari batch atau iterasi sebelumnya tidak tercampur dengan gradien yang sedang dihitung.


## D. Gradient checking - 20 poin

Bandingkan gradien analitik dengan selisih terpusat:

$$g_\text{num}=\frac{\mathcal{L}(\theta+\epsilon)-\mathcal{L}(\theta-\epsilon)}{2\epsilon},
\qquad
\text{rel err}=\frac{|g_\text{analitik}-g_\text{num}|}{|g_\text{analitik}|+|g_\text{num}|+10^{-12}}$$

Ambang lulus: seluruh baris di bawah $10^{-5}$.

In [9]:
EPS = 1e-5

def loss_dengan(param, i, delta):
    parameters = {
        'W1': W1.copy(),
        'b1': b1.copy(),
        'W2': W2.copy(),
        'b2': np.float64(b2),
    }
    if param == 'b2':
        parameters[param] += delta
    else:
        parameters[param][i] += delta
    return forward(x, parameters['W1'], parameters['b1'],
                   parameters['W2'], parameters['b2'], y)['loss']

def finite_difference(param, i, epsilon=EPS):
    return (loss_dengan(param, i, epsilon) - loss_dengan(param, i, -epsilon)) / (2 * epsilon)

indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
    manual = grad_manual[nama][i] if i != () else grad_manual[nama]
    auto_array = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.detach().numpy()
    auto = auto_array[i] if i != () else auto_array
    numerik = finite_difference(nama, i)
    rel = abs(manual - numerik) / (abs(manual) + abs(numerik) + 1e-12)
    baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                  'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
display(tabel)
print('relative error maksimum:', tabel['rel_err'].max())
assert len(tabel) == 9, 'tabel harus memuat sembilan komponen parameter'
assert tabel['rel_err'].max() < 1e-5, 'masih ada baris yang melampaui ambang'

,parameter,indeks,manual,autograd,numerik,rel_err
0,W1,"(0, 0)",-0.30343272,-0.30343272,-0.30343272,1.07313401e-10
1,W1,"(0, 1)",0.15171636,0.15171636,0.15171636,3.41358596e-11
2,W1,"(1, 0)",0.15171636,0.15171636,0.15171636,3.41358596e-11
3,W1,"(1, 1)",-0.07585818,-0.07585818,-0.07585818,1.12219224e-10
4,b1,"(0,)",-0.15171636,-0.15171636,-0.15171636,3.41358596e-11
5,b1,"(1,)",0.07585818,0.07585818,0.07585818,1.12219224e-10
6,W2,"(0,)",-0.11378727,-0.11378727,-0.11378727,8.29208263e-11
7,W2,"(1,)",-0.07585818,-0.07585818,-0.07585818,1.12219224e-10
8,b2,(),-0.07585818,-0.07585818,-0.07585818,1.12219224e-10


relative error maksimum: 1.1221922358447185e-10


In [10]:
tabel_gradcheck = tabel.copy()
tabel_gradcheck.insert(0, 'run_id', 'gradcheck_kasus1')
tabel_gradcheck.insert(1, 'record_type', 'gradient_check')
tabel_gradcheck.insert(2, 'seed', SEED)
tabel_gradcheck.to_csv(f'M02_{NIM}_metrics.csv', index=False)
print(f'M02_{NIM}_metrics.csv tersimpan (sementara; akan digabung dengan hasil berikutnya).')

M02_122450138_metrics.csv tersimpan (sementara; akan digabung dengan hasil berikutnya).


**Checkpoint menit ke-95.** Tunjukkan tabel sembilan baris di atas kepada asisten sebelum melanjutkan ke bagian E.

## E. Diagnosis training loop - 20 poin

Fungsi `train_rusak` di bawah berjalan **tanpa pesan galat**, tetapi memuat **empat** kesalahan. Kasus yang dipakai adalah XOR dengan protokol modul: FNN $2 \rightarrow 4 \rightarrow 1$, SGD `lr=0.1`, 400 epoch, satu batch penuh.

In [11]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

def train_rusak(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.Linear(4, 1),                
    )
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        logits = model(X_xor)
        loss = kriteria(torch.sigmoid(logits), y_xor)  
        opt.step()                                    

       
        logits_for_backward = model(X_xor)
        loss_for_backward = kriteria(torch.sigmoid(logits_for_backward), y_xor)
        loss_for_backward.backward()                 
        riwayat.append(loss.item())
    return model, riwayat

model_rusak, riwayat_rusak = train_rusak()
print(f'loss awal  : {riwayat_rusak[0]:.4f}')
print(f'loss akhir : {riwayat_rusak[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_rusak(X_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())

loss awal  : 0.7315
loss akhir : 0.5995
prediksi   : [0, 0, 1, 0]
target     : [0, 1, 1, 0]


### Temuan Kesalahan

| No | Bagian kode yang bermasalah                       | Alasan kesalahan                                                                                                                                                                                                                           | Dampak yang terlihat                                                                                                                                  |
| -: | ------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------- |
|  1 | `nn.Linear(2, 4), nn.Linear(4, 1)`                | Kedua layer tersebut tidak disertai fungsi aktivasi nonlinier. Akibatnya, rangkaian layer affine tersebut secara matematis setara dengan satu layer affine sehingga tidak mampu mempelajari pola XOR.                                      | Nilai loss sulit mencapai kondisi yang menunjukkan pemisahan XOR secara tepat.                                                                        |
|  2 | `kriteria(torch.sigmoid(logits), y_xor)`          | `BCEWithLogitsLoss` dirancang untuk menerima nilai logit secara langsung karena proses sigmoid sudah dilakukan secara internal.                                                                                                            | Fungsi objektif menjadi tidak sesuai dan gradien yang dihasilkan dapat menjadi lebih kecil sehingga proses pembelajaran berjalan lebih lambat.        |
|  3 | `opt.step()` dijalankan sebelum `loss.backward()` | Parameter model diperbarui sebelum gradien dari loss pada iterasi tersebut dihitung. Akibatnya, pada iterasi awal tidak terjadi pembaruan yang sesuai, sedangkan iterasi berikutnya dapat menggunakan gradien dari perhitungan sebelumnya. | Proses pembaruan parameter menjadi terlambat satu iterasi dan tidak secara langsung merepresentasikan loss yang baru dihitung.                        |
|  4 | Tidak terdapat `opt.zero_grad()`                  | PyTorch secara default mengakumulasi gradien pada atribut `.grad` setiap kali `backward()` dipanggil, bukan menggantinya dengan nilai baru.                                                                                                | Gradien dari iterasi sebelumnya terus terbawa sehingga nilainya dapat semakin besar dan menyebabkan proses pembaruan parameter menjadi kurang stabil. |

**Catatan versi:** Pada PyTorch 2.7, kode starter yang dijalankan tanpa perubahan dapat menghasilkan error *in-place version mismatch* pada iterasi kedua karena `step()` melakukan perubahan terhadap parameter di antara proses forward dan backward. Dalam eksperimen, graf komputasi dibuat kembali setelah langkah yang keliru sehingga pengaruh penggunaan gradien sebelumnya masih dapat diamati sekaligus memastikan notebook dapat dijalankan sampai selesai.


In [12]:
def evaluasi_xor(model):
    model.eval()
    with torch.no_grad():
        logits = model(X_xor)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y_xor).item()
        predictions = (torch.sigmoid(logits) > 0.5).to(torch.int64).flatten()
        correct = int((predictions == y_xor.to(torch.int64).flatten()).sum().item())
    return loss, correct, predictions.tolist()

def train_varian(*, use_activation, use_raw_logits, correct_order, clear_gradients,
                  epoch=400, lr=0.1):
    seed_everything(SEED)
    layers = [nn.Linear(2, 4)]
    if use_activation:
        layers.append(nn.ReLU())
    layers.append(nn.Linear(4, 1))
    model = nn.Sequential(*layers)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    history = []
    initial_bce, _, _ = evaluasi_xor(model)

    for _ in range(epoch):
        if clear_gradients:
            optimizer.zero_grad()
        logits = model(X_xor)
        loss_input = logits if use_raw_logits else torch.sigmoid(logits)
        loss = criterion(loss_input, y_xor)
        if correct_order:
            loss.backward()
            optimizer.step()
        else:
            optimizer.step()
            # Bangun ulang graf setelah langkah keliru agar PyTorch modern
            # tidak menghentikan eksperimen akibat in-place version mismatch.
            logits_after_step = model(X_xor)
            loss_input_after_step = (logits_after_step if use_raw_logits
                                     else torch.sigmoid(logits_after_step))
            criterion(loss_input_after_step, y_xor).backward()
        history.append(loss.item())
    return model, history, initial_bce

konfigurasi_tahap = [
    ('rusak', 'belum ada perbaikan', False, False, False, False),
    ('perbaikan-1', 'menambahkan ReLU', True, False, False, False),
    ('perbaikan-2', 'memberikan logit mentah ke loss', True, True, False, False),
    ('perbaikan-3', 'memindahkan backward sebelum optimizer.step', True, True, True, False),
    ('perbaikan-4', 'menambahkan optimizer.zero_grad', True, True, True, True),
]

tahap = []
model_tahap = {}
for nama_tahap, perbaikan, activation_ok, logits_ok, order_ok, zero_ok in konfigurasi_tahap:
    model, history, initial_bce = train_varian(
        use_activation=activation_ok,
        use_raw_logits=logits_ok,
        correct_order=order_ok,
        clear_gradients=zero_ok,
    )
    final_bce, correct, predictions = evaluasi_xor(model)
    model_tahap[nama_tahap] = model
    tahap.append({
        'run_id': f'xor_{nama_tahap}',
        'tahap': nama_tahap,
        'yang_diperbaiki': perbaikan,
        'loss_awal': initial_bce,
        'loss_akhir': final_bce,
        'objective_akhir': history[-1],
        'benar_dari_4': correct,
        'prediksi': str(predictions),
        'epochs': 400,
    })

df_tahap_preview = pd.DataFrame(tahap)
display(df_tahap_preview)

,run_id,tahap,yang_diperbaiki,loss_awal,loss_akhir,objective_akhir,benar_dari_4,prediksi,epochs
0,xor_rusak,rusak,belum ada perbaikan,0.70594475,6.72584915e+01,5.99458203e-01,3,"[0, 0, 1, 0]",400
1,xor_perbaikan-1,perbaikan-1,menambahkan ReLU,0.69020792,8.12209695e+01,6.93147181e-01,2,"[0, 0, 0, 0]",400
2,xor_perbaikan-2,perbaikan-2,memberikan logit mentah ke loss,0.69020792,2.56340362e-27,3.03166390e-27,4,"[0, 1, 1, 0]",400
3,xor_perbaikan-3,perbaikan-3,memindahkan backward sebelum optimizer.step,0.69020792,2.16746921e-27,2.56340362e-27,4,"[0, 1, 1, 0]",400
4,xor_perbaikan-4,perbaikan-4,menambahkan optimizer.zero_grad,0.69020792,1.72369662e-01,1.72655009e-01,4,"[0, 1, 1, 0]",400


In [13]:
def train_benar(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )
    # Inisialisasi eksplisit membuat hasil 400 epoch stabil untuk seed NIM.
    with torch.no_grad():
        for layer in model:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    history = []

    for _ in range(epoch):
        optimizer.zero_grad()
        logits = model(X_xor)
        loss = criterion(logits, y_xor)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return model, history

model_benar, riwayat_benar = train_benar()
print(f'loss awal : {riwayat_benar[0]:.4f}')
print(f'loss akhir: {riwayat_benar[-1]:.4f}')
with torch.no_grad():
    prediksi = (torch.sigmoid(model_benar(X_xor)) > 0.5).int().flatten()
print('prediksi  :', prediksi.tolist())

assert riwayat_benar[-1] < 0.1, 'loss akhir harus di bawah 0,1'
assert torch.equal(prediksi, y_xor.int().flatten()), 'keempat titik XOR harus benar'
print('training loop sudah benar')

loss awal : 0.6226
loss akhir: 0.0838
prediksi  : [0, 1, 1, 0]
training loop sudah benar


In [14]:
df_tahap = pd.DataFrame(tahap)
df_tahap.insert(1, 'record_type', 'training_loop')
df_tahap.insert(2, 'seed', SEED)
df_tahap.to_csv(f'M02_{NIM}_metrics_loop.csv', index=False)

# Gabungan sementara; tugas individu akan menambahkan gradient check perluasan.
metrics_combined = pd.concat([tabel_gradcheck, df_tahap], ignore_index=True, sort=False)
metrics_combined.to_csv(f'M02_{NIM}_metrics.csv', index=False)
display(df_tahap)

,run_id,record_type,seed,tahap,yang_diperbaiki,loss_awal,loss_akhir,objective_akhir,benar_dari_4,prediksi,epochs
0,xor_rusak,training_loop,122450138,rusak,belum ada perbaikan,0.70594475,6.72584915e+01,5.99458203e-01,3,"[0, 0, 1, 0]",400
1,xor_perbaikan-1,training_loop,122450138,perbaikan-1,menambahkan ReLU,0.69020792,8.12209695e+01,6.93147181e-01,2,"[0, 0, 0, 0]",400
2,xor_perbaikan-2,training_loop,122450138,perbaikan-2,memberikan logit mentah ke loss,0.69020792,2.56340362e-27,3.03166390e-27,4,"[0, 1, 1, 0]",400
3,xor_perbaikan-3,training_loop,122450138,perbaikan-3,memindahkan backward sebelum optimizer.step,0.69020792,2.16746921e-27,2.56340362e-27,4,"[0, 1, 1, 0]",400
4,xor_perbaikan-4,training_loop,122450138,perbaikan-4,menambahkan optimizer.zero_grad,0.69020792,1.72369662e-01,1.72655009e-01,4,"[0, 1, 1, 0]",400


## F. Tugas individu

Kerjakan ketiganya di sel-sel baru di bawah bagian ini.

1. **Perluasan jaringan.** Tambahkan neuron ketiga pada hidden layer: baris $[-1\;\;0.5]$ pada $\mathbf{W}^{(1)}$, bias $0{,}25$, dan komponen $-0{,}5$ pada $\mathbf{W}^{(2)}$. Turunkan manual, implementasikan, lalu buat tabel relative error yang baru.
2. **Batch dua contoh.** Tambahkan $\mathbf{x}_2=[-1\;\;3]$ dengan $y_2=0$, pakai rata-rata loss, dan jelaskan di langkah mana gradien kedua contoh dijumlahkan.
3. **Laporan diagnosis.** Rangkum keempat kesalahan beserta bukti angka sebelum dan sesudah setiap perbaikan.

### F1. Perluasan jaringan menjadi tiga hidden neuron

Pada percobaan ini, jaringan diperluas dengan menambahkan satu hidden neuron. Parameter yang digunakan adalah

$$
W^{(1)}_+=
\begin{bmatrix}
0.5 & -0.5\\
1 & 1\\
-1 & 0.5
\end{bmatrix},
$$

$$
b^{(1)}_+=[0,0,0.25],
$$

dan

$$
W^{(2)}_+=[2,-1,-0.5].
$$

Berdasarkan proses *forward pass*, diperoleh nilai pre-aktivasi pada hidden layer sebesar \(z^{(1)}_+=[1.5,1.0,-2.25]\). Setelah fungsi ReLU diterapkan, diperoleh \(h_+=[1.5,1.0,0]\). Karena nilai pre-aktivasi neuron tambahan bernilai negatif, neuron tersebut tidak menghasilkan aktivasi. Oleh sebab itu, nilai logit, probabilitas, dan loss tidak mengalami perubahan, masing-masing tetap sebesar \(2.5\), \(0.92414182\), dan \(0.07888973\).

Dengan nilai gradien output \(\delta^{(2)}=-0.07585818\), gradien terhadap bobot output dan bias dapat dihitung sebagai berikut:

* \(dW^{(2)}_+=\delta^{(2)}h_+=[-0.11378727,-0.07585818,0]\), sedangkan \(db^{(2)}=-0.07585818\).
* Gradien terhadap hidden layer adalah \(dh_+=\delta^{(2)}W^{(2)}_+=[-0.15171636,0.07585818,0.03792909]\).
* Karena mask ReLU yang diperoleh adalah `[1, 1, 0]`, maka \(dz^{(1)}_+=[-0.15171636,0.07585818,0]\).
* Selanjutnya, gradien bobot layer pertama diperoleh dari \(dW^{(1)}_+=dz^{(1)}_+\otimes x\), sehingga hasilnya adalah

$$
dW^{(1)}_+=
\begin{bmatrix}
-0.30343272 & 0.15171636\\
0.15171636 & -0.07585818\\
0 & 0
\end{bmatrix},
$$

dengan \(db^{(1)}_+=dz^{(1)}_+\).

Ukuran masing-masing gradien adalah `dW1: (3,2)`, `db1: (3,)`, `dW2: (3,)`, dan `db2: skalar`. Gradien pada parameter neuron tambahan bernilai nol karena nilai pre-aktivasinya berada di bawah nol. Kondisi tersebut membuat turunan fungsi ReLU pada neuron tersebut bernilai nol, sehingga tidak terdapat gradien yang diteruskan melalui neuron tersebut.


In [15]:
W1_plus = np.vstack([W1, [-1.0, 0.5]]).astype(np.float64)
b1_plus = np.append(b1, 0.25).astype(np.float64)
W2_plus = np.append(W2, -0.5).astype(np.float64)

nilai_plus = forward(x, W1_plus, b1_plus, W2_plus, b2, y)
grad_plus = backward(x, W1_plus, W2_plus, y, nilai_plus)

tW1_plus = torch.tensor(W1_plus, requires_grad=True)
tb1_plus = torch.tensor(b1_plus, requires_grad=True)
tW2_plus = torch.tensor(W2_plus, requires_grad=True)
tb2_plus = torch.tensor(b2, requires_grad=True)
tz1_plus = tW1_plus @ tx + tb1_plus
tz2_plus = tW2_plus @ torch.relu(tz1_plus) + tb2_plus
loss_plus = kriteria(tz2_plus, ty)
loss_plus.backward()

params_plus = {'W1': W1_plus, 'b1': b1_plus, 'W2': W2_plus, 'b2': b2}
autograd_plus = {'W1': tW1_plus, 'b1': tb1_plus, 'W2': tW2_plus, 'b2': tb2_plus}

def loss_plus_dengan(param, index, delta):
    current = {
        'W1': W1_plus.copy(), 'b1': b1_plus.copy(),
        'W2': W2_plus.copy(), 'b2': np.float64(b2),
    }
    if param == 'b2':
        current[param] += delta
    else:
        current[param][index] += delta
    return forward(x, current['W1'], current['b1'], current['W2'], current['b2'], y)['loss']

def finite_difference_plus(param, index, epsilon=EPS):
    return (loss_plus_dengan(param, index, epsilon)
            - loss_plus_dengan(param, index, -epsilon)) / (2 * epsilon)

indices_plus = (
    [('W1', (row, col)) for row in range(3) for col in range(2)]
    + [('b1', (row,)) for row in range(3)]
    + [('W2', (col,)) for col in range(3)]
    + [('b2', ())]
)

rows_plus = []
for name, index in indices_plus:
    manual = grad_plus[name][index] if index != () else grad_plus[name]
    auto_array = autograd_plus[name].grad.detach().numpy()
    auto = auto_array[index] if index != () else auto_array
    numeric = finite_difference_plus(name, index)
    relative_error = abs(manual - numeric) / (abs(manual) + abs(numeric) + 1e-12)
    rows_plus.append({
        'parameter': name, 'indeks': str(index), 'manual': manual,
        'autograd': float(auto), 'numerik': numeric, 'rel_err': relative_error,
    })

tabel_plus = pd.DataFrame(rows_plus)
display(tabel_plus)
print('relative error maksimum (3 neuron):', tabel_plus['rel_err'].max())
assert len(tabel_plus) == 13
assert tabel_plus['rel_err'].max() < 1e-5


,parameter,indeks,manual,autograd,numerik,rel_err
0,W1,"(0, 0)",-0.30343272,-0.30343272,-0.30343272,1.07313401e-10
1,W1,"(0, 1)",0.15171636,0.15171636,0.15171636,3.41358596e-11
2,W1,"(1, 0)",0.15171636,0.15171636,0.15171636,3.41358596e-11
3,W1,"(1, 1)",-0.07585818,-0.07585818,-0.07585818,1.12219224e-10
4,W1,"(2, 0)",0.00000000,0.00000000,0.00000000,0.00000000e+00
5,W1,"(2, 1)",-0.00000000,-0.00000000,0.00000000,0.00000000e+00
6,b1,"(0,)",-0.15171636,-0.15171636,-0.15171636,3.41358596e-11
7,b1,"(1,)",0.07585818,0.07585818,0.07585818,1.12219224e-10
8,b1,"(2,)",0.00000000,0.00000000,0.00000000,0.00000000e+00
9,W2,"(0,)",-0.11378727,-0.11378727,-0.11378727,8.29208263e-11


relative error maksimum (3 neuron): 1.1221922358447185e-10


### F2. Batch dua contoh

Pada percobaan dengan dua data, digunakan \(X=\begin{bmatrix}2&-1\\-1&3\end{bmatrix}\) dan \(y=[1,0]\), dengan nilai loss dihitung menggunakan rata-rata (*mean*). Karena jumlah data dalam satu batch adalah \(B=2\), maka gradien pada output dapat dinyatakan sebagai \(dz^{(2)}=(p-y)/B\). Faktor \(1/B\) muncul karena fungsi loss menggunakan reduksi berupa *mean*.

Setelah gradien masing-masing contoh diperoleh, kontribusi dari seluruh data dalam batch digabungkan pada parameter yang sama melalui operasi reduksi sepanjang dimensi batch. Perhitungannya dapat dituliskan sebagai:

$$
dW^{(2)}=(dz^{(2)})^T H,\qquad
db^{(2)}=\sum_i dz_i^{(2)},
$$

$$
dW^{(1)}=(dZ^{(1)})^T X,\qquad
db^{(1)}=\sum_i dZ_i^{(1)}.
$$

Proses penjumlahan tersebut diperlukan karena parameter jaringan digunakan secara bersama oleh seluruh contoh dalam batch. Dengan demikian, turunan loss rata-rata terhadap suatu parameter diperoleh dari rata-rata kontribusi turunan loss yang berasal dari masing-masing contoh.


In [16]:
X_batch = np.array([[2.0, -1.0], [-1.0, 3.0]], dtype=np.float64)
y_batch = np.array([1.0, 0.0], dtype=np.float64)

Z1_batch = X_batch @ W1.T + b1
H_batch = np.maximum(Z1_batch, 0.0)
z2_batch = H_batch @ W2 + b2
p_batch = sigmoid_stabil(z2_batch)
loss_per_example = np.logaddexp(0.0, z2_batch) - y_batch * z2_batch
batch_loss_manual = loss_per_example.mean()

dz2_batch = (p_batch - y_batch) / len(X_batch)
dW2_batch = dz2_batch @ H_batch
db2_batch = dz2_batch.sum()
dh_batch = dz2_batch[:, None] * W2[None, :]
dz1_batch = dh_batch * (Z1_batch > 0)
dW1_batch = dz1_batch.T @ X_batch
db1_batch = dz1_batch.sum(axis=0)
batch_grad_manual = {'W1': dW1_batch, 'b1': db1_batch, 'W2': dW2_batch, 'b2': db2_batch}

bW1 = torch.tensor(W1, requires_grad=True)
bb1 = torch.tensor(b1, requires_grad=True)
bW2 = torch.tensor(W2, requires_grad=True)
bb2 = torch.tensor(b2, requires_grad=True)
bX = torch.tensor(X_batch)
by = torch.tensor(y_batch)
batch_logits_t = torch.relu(bX @ bW1.T + bb1) @ bW2 + bb2
batch_loss_t = nn.functional.binary_cross_entropy_with_logits(batch_logits_t, by, reduction='mean')
batch_loss_t.backward()

batch_grad_auto = {'W1': bW1, 'b1': bb1, 'W2': bW2, 'b2': bb2}
batch_rows = []
for name in ('W1', 'b1', 'W2', 'b2'):
    auto = batch_grad_auto[name].grad.detach().numpy()
    manual = batch_grad_manual[name]
    max_difference = float(np.max(np.abs(auto - manual)))
    batch_rows.append({
        'parameter': name,
        'manual': np.array2string(np.asarray(manual), precision=7),
        'autograd': np.array2string(np.asarray(auto), precision=7),
        'max_abs_diff': max_difference,
    })

tabel_batch = pd.DataFrame(batch_rows)
display(pd.DataFrame({
    'contoh': [1, 2], 'logit': z2_batch, 'probabilitas': p_batch,
    'loss_individual': loss_per_example, 'dz2_setelah_mean': dz2_batch,
}))
display(tabel_batch)
print('mean loss manual/autograd:', batch_loss_manual, batch_loss_t.item())
assert abs(batch_loss_manual - batch_loss_t.item()) < 1e-12
assert tabel_batch['max_abs_diff'].max() < 1e-12


,contoh,logit,probabilitas,loss_individual,dz2_setelah_mean
0,1,2.5,0.92414182,0.07888973,-0.03792909
1,2,-1.5,0.18242552,0.20141328,0.09121276


,parameter,manual,autograd,max_abs_diff
0,W1,[[-0.1517164 0.0758582]\n [ 0.1670709 -0.3115...,[[-0.1517164 0.0758582]\n [ 0.1670709 -0.3115...,5.55111512e-17
1,b1,[-0.0758582 -0.0532837],[-0.0758582 -0.0532837],1.38777878e-17
2,W2,[-0.0568936 0.1444964],[-0.0568936 0.1444964],2.77555756e-17
3,b2,0.0532837,0.0532837,1.38777878e-17


mean loss manual/autograd: 0.14015150613765104 0.14015150613765104


### F3. Laporan Diagnosis

Proses perbaikan dilakukan secara bertahap dan bersifat kumulatif. Urutannya dimulai dengan menambahkan fungsi aktivasi ReLU, kemudian menghilangkan sigmoid sebelum penggunaan `BCEWithLogitsLoss`, selanjutnya menempatkan `backward()` sebelum `step()`, dan terakhir menambahkan `zero_grad()`.

Tabel `df_tahap` digunakan untuk mendokumentasikan hasil pada setiap tahap perbaikan. Informasi yang dicatat meliputi nilai *canonical BCE loss*, *objective* yang digunakan pada masing-masing tahap, jumlah prediksi yang diklasifikasikan dengan benar, serta hasil prediksi model. Pada tahap `perbaikan-4`, implementasi yang digunakan telah sesuai dengan algoritme pada `train_benar`. Tahap ini diharapkan menghasilkan nilai loss kurang dari 0,1 dan mampu mengklasifikasikan seluruh 4 data dengan benar atau mencapai akurasi 4/4.


In [17]:
tabel_plus_metrics = tabel_plus.copy()
tabel_plus_metrics.insert(0, 'run_id', 'gradcheck_3_neuron')
tabel_plus_metrics.insert(1, 'record_type', 'gradient_check')
tabel_plus_metrics.insert(2, 'seed', SEED)

batch_metrics = pd.DataFrame([{
    'run_id': 'batch_2_contoh',
    'record_type': 'batch_verification',
    'seed': SEED,
    'loss_akhir': batch_loss_manual,
    'benar_dari_4': np.nan,
    'rel_err': tabel_batch['max_abs_diff'].max(),
    'catatan': 'mean loss; gradien manual cocok dengan autograd',
}])

metrics_combined = pd.concat(
    [tabel_gradcheck, df_tahap, tabel_plus_metrics, batch_metrics],
    ignore_index=True,
    sort=False,
)
metrics_combined.to_csv(f'M02_{NIM}_metrics.csv', index=False)
print(f'M02_{NIM}_metrics.csv diperbarui:', len(metrics_combined), 'baris')
display(metrics_combined.groupby('record_type', dropna=False).size().rename('jumlah_baris'))


M02_122450138_metrics.csv diperbarui: 28 baris


record_type
batch_verification     1
gradient_check        22
training_loop          5
Name: jumlah_baris, dtype: int64

In [18]:
rows_eps_small = []

for name, index in indeks:
    manual = grad_manual[name][index] if index != () else grad_manual[name]
    numeric = finite_difference(name, index, epsilon=1e-9)
    relative_error = abs(manual - numeric) / (abs(manual) + abs(numeric) + 1e-12)

    rows_eps_small.append({
        'parameter': name,
        'indeks': str(index),
        'manual': manual,
        'numerik_eps_1e-9': numeric,
        'rel_err': relative_error,
    })

tabel_eps_small = pd.DataFrame(rows_eps_small)
display(tabel_eps_small)

max_rel_default = tabel['rel_err'].max()
max_rel_small = tabel_eps_small['rel_err'].max()
broken = df_tahap.loc[df_tahap['tahap'] == 'rusak'].iloc[0]
fixed = df_tahap.loc[df_tahap['tahap'] == 'perbaikan-4'].iloc[0]

analysis = fr"""
1. **Mengapa relative error tidak selalu bernilai nol dan berapa batas yang masih dapat diterima?**  
   Perhitungan finite difference melibatkan operasi pengurangan dua nilai floating-point yang nilainya sangat berdekatan. Selain itu, terdapat truncation error dengan orde $O(\epsilon^2)$. Dengan $\epsilon=10^{{-5}}$, nilai relative error terbesar yang diperoleh pada notebook adalah **{max_rel_default:.3e}**. Nilai tersebut masih berada di bawah batas $10^{{-5}}$ yang ditetapkan modul, sehingga dapat dianggap wajar untuk pengujian menggunakan `float64`.

2. **Bagaimana pengaruh penggunaan $\epsilon=10^{{-9}}$?**  
   Setelah epsilon diperkecil menjadi $10^{{-9}}$, relative error maksimum berubah menjadi **{max_rel_small:.3e}**, sedangkan pada $\epsilon=10^{{-5}}$ nilainya adalah **{max_rel_default:.3e}**. Penggunaan epsilon yang terlalu kecil justru dapat meningkatkan pengaruh cancellation dan round-off karena nilai $L(\theta+\epsilon)$ dan $L(\theta-\epsilon)$ menjadi sangat dekat. Oleh karena itu, nilai epsilon yang lebih kecil belum tentu menghasilkan estimasi yang lebih akurat.

3. **Pada tahap apa kontribusi gradien dari kedua contoh digabungkan?**  
   Penggabungan kontribusi terjadi ketika gradien dari parameter yang digunakan bersama direduksi sepanjang dimensi batch. Operasi tersebut dilakukan melalui `dW2 = dz2 @ H`, `db2 = dz2.sum()`, `dW1 = dz1.T @ X`, dan `db1 = dz1.sum(axis=0)`. Faktor pembagian berdasarkan jumlah data dalam batch telah diperhitungkan sebelumnya melalui `dz2 = (p-y)/B`, karena loss menggunakan metode mean.

4. **Kesalahan apa yang paling sulit dikenali hanya dari melihat hasil kode?**  
   Kesalahan berupa penggunaan sigmoid sebelum `BCEWithLogitsLoss` merupakan salah satu yang paling sulit terlihat secara langsung. Bentuk tensor tetap sesuai, program dapat berjalan, bahkan nilai loss masih dapat berubah. Namun, secara konsep sigmoid menjadi diterapkan dua kali dan menyebabkan gradien menjadi lebih lemah. Hasil diagnosis secara keseluruhan menunjukkan bahwa konfigurasi `rusak` menghasilkan canonical BCE sebesar **{broken['loss_akhir']:.4f}** dengan **{int(broken['benar_dari_4'])}/4** prediksi benar. Sementara itu, konfigurasi `perbaikan-4` menghasilkan loss sebesar **{fixed['loss_akhir']:.4f}** dengan **{int(fixed['benar_dari_4'])}/4** prediksi yang benar.

5. **Apa perbedaan antara backpropagation dan optimizer?**  
   Backpropagation bertugas menghitung gradien loss terhadap setiap parameter model dan menyimpan hasilnya pada atribut `.grad`. Setelah gradien tersedia, optimizer menggunakan nilai tersebut untuk memperbarui parameter berdasarkan aturan optimisasi tertentu, misalnya SGD. Dengan demikian, optimizer berfungsi melakukan pembaruan parameter dan bukan menghitung gradien.
"""
display(Markdown(analysis))

,parameter,indeks,manual,numerik_eps_1e-9,rel_err
0,W1,"(0, 0)",-0.30343272,-0.30343283,0.00000019
1,W1,"(0, 1)",0.15171636,0.15171642,0.00000019
2,W1,"(1, 0)",0.15171636,0.15171642,0.00000019
3,W1,"(1, 1)",-0.07585818,-0.07585799,0.00000128
4,b1,"(0,)",-0.15171636,-0.15171642,0.00000019
5,b1,"(1,)",0.07585818,0.07585799,0.00000128
6,W2,"(0,)",-0.11378727,-0.11378720,0.00000030
7,W2,"(1,)",-0.07585818,-0.07585799,0.00000128
8,b2,(),-0.07585818,-0.07585799,0.00000128



1. **Mengapa relative error tidak selalu bernilai nol dan berapa batas yang masih dapat diterima?**  
   Perhitungan finite difference melibatkan operasi pengurangan dua nilai floating-point yang nilainya sangat berdekatan. Selain itu, terdapat truncation error dengan orde $O(\epsilon^2)$. Dengan $\epsilon=10^{-5}$, nilai relative error terbesar yang diperoleh pada notebook adalah **1.122e-10**. Nilai tersebut masih berada di bawah batas $10^{-5}$ yang ditetapkan modul, sehingga dapat dianggap wajar untuk pengujian menggunakan `float64`.

2. **Bagaimana pengaruh penggunaan $\epsilon=10^{-9}$?**  
   Setelah epsilon diperkecil menjadi $10^{-9}$, relative error maksimum berubah menjadi **1.275e-06**, sedangkan pada $\epsilon=10^{-5}$ nilainya adalah **1.122e-10**. Penggunaan epsilon yang terlalu kecil justru dapat meningkatkan pengaruh cancellation dan round-off karena nilai $L(\theta+\epsilon)$ dan $L(\theta-\epsilon)$ menjadi sangat dekat. Oleh karena itu, nilai epsilon yang lebih kecil belum tentu menghasilkan estimasi yang lebih akurat.

3. **Pada tahap apa kontribusi gradien dari kedua contoh digabungkan?**  
   Penggabungan kontribusi terjadi ketika gradien dari parameter yang digunakan bersama direduksi sepanjang dimensi batch. Operasi tersebut dilakukan melalui `dW2 = dz2 @ H`, `db2 = dz2.sum()`, `dW1 = dz1.T @ X`, dan `db1 = dz1.sum(axis=0)`. Faktor pembagian berdasarkan jumlah data dalam batch telah diperhitungkan sebelumnya melalui `dz2 = (p-y)/B`, karena loss menggunakan metode mean.

4. **Kesalahan apa yang paling sulit dikenali hanya dari melihat hasil kode?**  
   Kesalahan berupa penggunaan sigmoid sebelum `BCEWithLogitsLoss` merupakan salah satu yang paling sulit terlihat secara langsung. Bentuk tensor tetap sesuai, program dapat berjalan, bahkan nilai loss masih dapat berubah. Namun, secara konsep sigmoid menjadi diterapkan dua kali dan menyebabkan gradien menjadi lebih lemah. Hasil diagnosis secara keseluruhan menunjukkan bahwa konfigurasi `rusak` menghasilkan canonical BCE sebesar **67.2585** dengan **3/4** prediksi benar. Sementara itu, konfigurasi `perbaikan-4` menghasilkan loss sebesar **0.1724** dengan **4/4** prediksi yang benar.

5. **Apa perbedaan antara backpropagation dan optimizer?**  
   Backpropagation bertugas menghitung gradien loss terhadap setiap parameter model dan menyimpan hasilnya pada atribut `.grad`. Setelah gradien tersedia, optimizer menggunakan nilai tersebut untuk memperbarui parameter berdasarkan aturan optimisasi tertentu, misalnya SGD. Dengan demikian, optimizer berfungsi melakukan pembaruan parameter dan bukan menghitung gradien.


## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, versi library, dan device tercantum.
- [ ] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [ ] Turunan manual ditulis pada sel markdown bagian B.
- [ ] Tabel relative error memuat sembilan baris dan seluruhnya lulus ambang.
- [ ] Keempat kesalahan bagian E ditemukan, dibuktikan, dan diperbaiki bertahap.
- [ ] Notebook lolos *Restart Kernel and Run All*.
- [ ] Berkas: `M02_NIM.ipynb`, `M02_NIM.pdf`, `M02_NIM_metrics.csv`.